## Version note
- modified version fixes C-side DB filename mismatch: output is `thailand_trip_full.db`.
- fixes attraction ticket lookup: uses `cost_item_master.related_attraction_id`, not missing `cost_item_attraction_map`.
- adds `member_a_record_export` and `c_cost_coverage_summary` views for A-side cost validation.
- adds transparent imputed costs for market/activity records so demo budget is not silently 0.


## 0. 安裝套件

In [3]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q google-adk litellm
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q sentence-transformers faiss-cpu
!pip install -q openpyxl  # 讀取 .xlsx 必要套件
!pip install -q nest_asyncio
print('✅ 所有套件安裝完成')

'pip' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC
'pip' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC
'pip' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC
'pip' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC
'pip' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


✅ 所有套件安裝完成


'pip' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


## 1. 環境設定：上傳 Excel + 載入 API Key

In [4]:

from pathlib import Path
import os

# C 端輸入資料：支援 Colab 上傳，也支援本機專案資料夾搜尋
EXCEL_GLOB_PATTERNS = [
    '*DATA_FINAL*.xlsx',
    '*生成式DATA_FINAL*.xlsx',
]

try:
    from google.colab import files
    print('Please upload the DATA_FINAL xlsx file')
    uploaded = files.upload()
    EXCEL_FILENAME = list(uploaded.keys())[0]
    EXCEL_PATH = f'/content/{EXCEL_FILENAME}'
except ModuleNotFoundError:
    candidates = []
    for pattern in EXCEL_GLOB_PATTERNS:
        candidates.extend(Path.cwd().glob(pattern))
        candidates.extend(Path.cwd().glob(f'**/{pattern}'))
        candidates.extend((Path.home() / 'Downloads').glob(pattern))

    matched = next((p for p in candidates if p.exists() and not p.name.startswith('~$')), None)
    if matched is None:
        raise FileNotFoundError(
            'Cannot find DATA_FINAL xlsx. Put it next to the notebook, under the project folder, or in Downloads.'
        )
    EXCEL_PATH = str(matched)

print(f'Excel path: {EXCEL_PATH}')


Excel path: C:\Users\h\Desktop\codex\泰國\C給A\生成式DATA_FINAL.xlsx


In [5]:
import os, nest_asyncio
nest_asyncio.apply()

def load_groq_api_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get('GROQ_API_KEY')
    except Exception:
        pass
    if not key:
        key = os.getenv('GROQ_API_KEY')
    if not key:
        import getpass
        key = getpass.getpass('請輸入 GROQ_API_KEY：')
    if not key:
        raise RuntimeError('找不到 GROQ_API_KEY')
    return key.strip()

GROQ_API_KEY = load_groq_api_key()
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
GROQ_MODEL = 'groq/llama-3.3-70b-versatile'
GROQ_MODEL_LIGHT = 'groq/llama-3.1-8b-instant'
print(f'✅ GROQ_API_KEY 載入完成，使用模型：{GROQ_MODEL}')

KeyboardInterrupt: Interrupted by user

---
# Part 1：讀取 Excel → 寫入 SQLite

### 讀取邏輯說明
Excel 裡的每一個 sheet 名稱對應一張 table（例如 `attraction_master`、`restaurant_master`）。
我們使用 `pandas.read_excel(sheet_name='...')` 讀取，再寫進 SQLite。

### 核心欄位對照（票價查詢用）
- 景點票價不在 `attraction_master` 裡，而是在 `cost_item_master` 的 `cost_category = 'attraction_ticket'`
- 住宿費用在 `cost_item_master` 的 `cost_category = 'accommodation'`
- `typical_price` 是建議使用的代表價格欄位

In [1]:

import pandas as pd
import sqlite3
import pathlib
from datetime import datetime

# A 端 run_member_a_api.py 會搜尋 **/thailand_trip_full.db
DB_PATH = pathlib.Path('/content/thailand_trip_full.db')
if not pathlib.Path('/content').exists():
    DB_PATH = pathlib.Path('thailand_trip_full.db')

xl = pd.ExcelFile(EXCEL_PATH)
print(f'Excel sheet count: {len(xl.sheet_names)}')
for i, name in enumerate(xl.sheet_names):
    print(f'  [{i+1:02d}] {name}')


NameError: name 'EXCEL_PATH' is not defined

In [2]:
TARGET_SHEETS = xl.sheet_names

conn = sqlite3.connect(DB_PATH)

loaded_dfs = {}

for sheet in TARGET_SHEETS:

    matched = [s for s in xl.sheet_names if s.strip().lower() == sheet.lower()]
    if not matched:
        print(f'⚠️  找不到 sheet: {sheet}，請確認 Excel 中的 sheet 名稱')
        continue

    actual_name = matched[0]
    df = pd.read_excel(EXCEL_PATH, sheet_name=actual_name)

    df.columns = [c.strip().lower() for c in df.columns]

    df.to_sql(sheet, conn, if_exists='replace', index=False)

    loaded_dfs[sheet] = df
    print(f'✅ {sheet}：{len(df)} 筆，{len(df.columns)} 欄')

conn.commit()
print(f'\n✅ 全部寫入 SQLite 完成，資料庫路徑：{DB_PATH}')

NameError: name 'xl' is not defined

In [6]:
for sheet, df in loaded_dfs.items():
    print(f'\n=== {sheet} ===')
    print('欄位：', list(df.columns[:8]), '...')
    display(df.head(2))


=== attraction_master ===
欄位： ['attraction_id', 'attraction_name_zh', 'attraction_name_en', 'city_id', 'area_id', 'attraction_type', 'indoor_outdoor', 'short_description'] ...


,attraction_id,attraction_name_zh,attraction_name_en,city_id,area_id,attraction_type,indoor_outdoor,short_description,main_features,unique_value_for_agent,...,elderly_friendly,accessibility_note,dress_code_or_rules,safety_notice,common_issues,recommended_reason,not_recommended_for,data_source_name,source_url,last_checked_date
0,A001,大皇宮,The Grand Palace,C001,AREA004,皇室建築、寺廟,室外為主,曼谷皇室建築群核心，集合宮殿、寺廟與傳統泰式建築細節。,皇室建築群；玉佛寺；金色尖塔與壁畫；老城區文化路線,判斷曼谷文化行程核心點，需結合服裝規範與高溫人潮限制。,...,不太適合長時間步行長輩,園區大、步行與曝曬多，行動不便者需預留休息。,寺廟區需遮肩與過膝，避免無袖、短褲。,注意高溫補水與門口票務/導遊招攬。,服裝不合可能被要求租借；尖峰團客多。,能展現曼谷皇室與佛教建築的代表性，最適合作為老城區文化半日行程起點。,討厭人潮、無法接受服裝限制、只想逛街者,The Grand Palace Official Website,https://www.royalgrandpalace.th/,2026-05-23
1,A002,玉佛寺,Wat Phra Kaew,C001,AREA004,寺廟,混合,供奉玉佛的泰國重要寺廟，與大皇宮同區但宗教意義更強。,玉佛殿；拉瑪堅壁畫；金色佛塔；宮殿園區,可作為宗教禮儀與服裝規則判斷的高權重節點。,...,部分適合,需排隊與脫鞋，長輩行程需降低後續步行量。,嚴格寺廟服裝規定，殿內拍照限制需遵守。,人多時保管隨身物品，遵守殿內動線。,不可把它當成一般拍照景點，宗教規範較多。,與大皇宮連動性強，可讓 AI 把兩者合併而非拆成兩個遠距點。,只想短停留或不願遵守寺廟規範者,The Grand Palace Official Website,https://www.royalgrandpalace.th/,2026-05-23



=== restaurant_master ===
欄位： ['restaurant_id', 'restaurant_name_zh', 'restaurant_name_en', 'city_id', 'area_id', 'primary_transport_mode_ids', 'nearest_hub_id', 'food_category'] ...


,restaurant_id,restaurant_name_zh,restaurant_name_en,city_id,area_id,primary_transport_mode_ids,nearest_hub_id,food_category,cuisine_type,dining_style,...,allergy_notice,family_friendly,elderly_friendly,accessibility_note,hygiene_or_safety_notice,recommended_reason,not_recommended_for,data_source_name,source_url,last_checked_date
0,B001,Thipsamai Pad Thai,Thipsamai Pad Thai,C001,AREA004,TRANS004;TRANS001;TRANS002,HUB021,米其林推薦/經典小吃,泰式炒河粉,內用餐廳/外帶,...,麩質需注意,是,普通；需考量排隊、座位與步行距離,Old Town / Pratu Phi周邊需確認實際入口與座位配置；若為市場/街邊/半戶外...,營業時間、冰塊、熟食保存與過敏原需以現場或官方資訊確認。,老城文化行程後的經典Pad Thai節點；招牌如 Pad Thai Sen Jan 能讓行程...,趕時間、花生或蝦過敏者,Michelin Guide; Google Maps,https://guide.michelin.com/en/search?type=rest...,2026-05-23
1,B002,朱拉豬肉媽媽麵,Jeh O Chula,C001,AREA001,TRANS004;TRANS001;TRANS002,HUB018,米其林推薦/宵夜名店,泰式酸辣麵鍋,內用餐廳,...,麩質需注意,是,普通；需考量排隊、座位與步行距離,Banthat Thong / Chula周邊需確認實際入口與座位配置；若為市場/街邊/半戶...,營業時間、冰塊、熟食保存與過敏原需以現場或官方資訊確認。,Siam購物後的深夜美食收尾；招牌如 Mama Tom Yum 能讓行程不是只停留在景點清單...,怕辣、帶幼兒、不能等待者,Michelin Guide; Google Maps,https://guide.michelin.com/en/search?type=rest...,2026-05-23



=== cost_item_master ===
欄位： ['cost_item_id', 'cost_category', 'cost_subcategory', 'item_name_zh', 'item_name_en', 'city_id', 'area_id', 'related_attraction_id'] ...


,cost_item_id,cost_category,cost_subcategory,item_name_zh,item_name_en,city_id,area_id,related_attraction_id,related_transport_mode_id,unit,...,suitable_user_types,seasonality_note,booking_required,included_items,excluded_items,cost_estimation_note,common_issues,data_source_name,source_url,last_checked_date
0,COST001,attraction_ticket,palace_ticket,大皇宮外國成人門票,Grand Palace foreign adult ticket,C001,AREA004,A001,not_applicable,per_person,...,文化旅客;第一次曼谷;長輩,全年固定性較高，但特殊活動可能調整開放區域,否,大皇宮、玉佛寺、紡織博物館等官方票券包含範圍,不含導遊、交通、服裝租借,估算老城區文化路線時應作為高門票景點核心項目；低預算行程若加入此點，當日其他景點可搭配免費寺...,外國人票價較高；服裝規定嚴格；閉館前入場時間有限。,Grand Palace Official,https://www.royalgrandpalace.th/en/visit/pract...,2026-05-23
1,COST002,attraction_ticket,temple_ticket,鄭王廟外國成人門票,Wat Arun foreign adult ticket,C001,AREA005,A004,not_applicable,per_person,...,文化旅客;拍照旅客;河岸行程,全年大致穩定，雨天登塔與拍照體驗下降,否,寺廟參觀門票,不含渡船、傳統服裝租借,適合估算昭披耶河岸半日遊費用，可與臥佛寺、大皇宮合併成文化路線；若搭渡船需另加短程船費。,中午炎熱；階梯與地面濕滑時不利長輩。,Tourism Authority of Thailand,https://www.tourismthailand.org/Attraction/wat...,2026-05-23



=== accommodation_area_master ===
欄位： ['accommodation_area_id', 'area_id', 'accommodation_level', 'suitable_user_types', 'strengths', 'weaknesses', 'recommended_stay_reason', 'not_recommended_for'] ...


,accommodation_area_id,area_id,accommodation_level,suitable_user_types,strengths,weaknesses,recommended_stay_reason,not_recommended_for,data_source_name,source_url,last_checked_date
0,ACCOM001,AREA001,medium,first_time;shopping;couple;family,BTS 與大型商場集中，雨天備案多，餐飲選擇密集。,房價相對較高，尖峰時段人潮多。,適合作為首次曼谷自由行的核心住宿區。,極低預算或偏好安靜度假者。,internal_accommodation_structuring,general,2026-05-23
1,ACCOM002,AREA002,medium_high,couple;business;nightlife;comfort,餐廳、咖啡廳與夜生活選擇多，BTS 沿線住宿彈性高。,道路塞車明顯，部分區域夜間較吵。,適合多日曼谷自由行與偏好餐飲夜生活者。,非常害怕塞車或偏好安靜家庭住宿者。,internal_accommodation_structuring,general,2026-05-23



=== area_master ===
欄位： ['area_id', 'city_id', 'city_name_en', 'city_name_zh', 'area_name_en', 'area_name_zh', 'region_group', 'area_type'] ...


,area_id,city_id,city_name_en,city_name_zh,area_name_en,area_name_zh,region_group,area_type,walkability_level,traffic_congestion_level,rainy_day_suitable,recommended_stay_area,area_description,data_source_name,source_url,last_checked_date
0,AREA001,C001,Bangkok,曼谷,Siam,暹羅,Central Thailand,shopping_area,high,high,suitable,1,曼谷購物、百貨與 BTS 轉乘核心，適合作為雨天備案、購物與餐飲集中區。,Tourism Authority of Thailand,https://www.tourismthailand.org/,2026-05-23
1,AREA002,C001,Bangkok,曼谷,Sukhumvit,素坤逸,Central Thailand,business_dining_area,medium,high,partially_suitable,1,沿 BTS 展開的住宿、餐飲與夜生活帶，適合多日自由行與夜間活動，但道路尖峰塞車明顯。,Tourism Authority of Thailand,https://www.tourismthailand.org/,2026-05-23


In [7]:

# ===== C 端資料品質與 A 端串接 views =====
# 重點：
# 1. attraction 票價直接從 cost_item_master.related_attraction_id 查。
# 2. market / activity 原始表通常沒有價格，這裡建立透明的估價規則與 export view。
# 3. A 若要更穩，可以改讀 member_a_record_export；目前也可用這些 view 做驗證。

conn.execute("DROP VIEW IF EXISTS attraction_ticket_view")
conn.execute(
    """
    CREATE VIEW attraction_ticket_view AS
    SELECT
        c.cost_item_id,
        c.item_name_zh,
        c.item_name_en,
        c.related_attraction_id,
        c.city_id,
        c.area_id,
        c.unit,
        c.currency,
        c.min_price,
        c.typical_price,
        c.max_price,
        c.budget_level,
        c.suitable_user_types,
        c.cost_estimation_note
    FROM cost_item_master c
    WHERE c.cost_category = 'attraction_ticket'
      AND c.related_attraction_id IS NOT NULL
      AND c.related_attraction_id != ''
    """
)

conn.execute("DROP VIEW IF EXISTS accommodation_cost_view")
conn.execute(
    """
    CREATE VIEW accommodation_cost_view AS
    SELECT
        c.cost_item_id,
        c.item_name_zh,
        c.item_name_en,
        c.city_id,
        c.area_id,
        c.unit,
        c.currency,
        c.min_price,
        c.typical_price,
        c.max_price,
        c.budget_level,
        c.suitable_user_types,
        c.cost_estimation_note
    FROM cost_item_master c
    WHERE c.cost_category = 'accommodation'
    """
)

conn.execute("""
CREATE TABLE IF NOT EXISTS c_data_quality_notes (
    note_id TEXT PRIMARY KEY,
    area TEXT,
    issue TEXT,
    fix_summary TEXT,
    created_at TEXT
)
""")

quality_notes = [
    (
        'C_NOTE_001',
        'cost',
        'market_master 與 activity_master 原始資料多數沒有直接票價欄位，A 端若直接讀取會得到 0 THB。',
        '新增 c_cost_imputation_reference 與 member_a_record_export view，提供 estimated_cost_thb 與 cost_quality。'
    ),
    (
        'C_NOTE_002',
        'notebook',
        '原 query_attraction_cost 使用 cost_item_attraction_map，但目前 DB schema 沒有這張表。',
        '改成直接使用 cost_item_master.related_attraction_id 查詢票價。'
    ),
    (
        'C_NOTE_003',
        'db_path',
        '原 notebook 輸出 thailand_trip.db，與 A 端尋找 thailand_trip_full.db 的命名不一致。',
        '修改 notebook 預設輸出 thailand_trip_full.db。'
    ),
]
conn.executemany(
    """
    INSERT OR REPLACE INTO c_data_quality_notes
    (note_id, area, issue, fix_summary, created_at)
    VALUES (?, ?, ?, ?, ?)
    """,
    [(*row, datetime.now().isoformat(timespec='seconds')) for row in quality_notes]
)

conn.execute("""
CREATE TABLE IF NOT EXISTS c_cost_imputation_reference (
    ref_id TEXT PRIMARY KEY,
    entity_category TEXT NOT NULL,
    match_key TEXT NOT NULL,
    estimated_cost_thb REAL NOT NULL,
    cost_note TEXT NOT NULL,
    priority INTEGER NOT NULL DEFAULT 100
)
""")
conn.execute("DELETE FROM c_cost_imputation_reference")
conn.executemany(
    """
    INSERT INTO c_cost_imputation_reference
    (ref_id, entity_category, match_key, estimated_cost_thb, cost_note, priority)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    [
        ('REF_MARKET_GENERAL', 'market', 'general', 300, '市集/商圈通常免門票，但抓餐飲與小額購物基本消費 300 THB/人。', 900),
        ('REF_MARKET_NIGHT', 'market', 'night', 450, '夜市通常免門票，但晚餐、小吃與飲料抓 450 THB/人。', 100),
        ('REF_MARKET_SHOPPING', 'market', 'shopping', 500, '購物商場/市集抓餐飲與基本購物消費 500 THB/人；精品購物不含在內。', 200),
        ('REF_ACTIVITY_GENERAL', 'activity', 'general', 800, '活動資料未建立明確票價時，以一般體驗活動 800 THB/人估算。', 900),
        ('REF_ACTIVITY_SPA', 'activity', 'spa', 800, '泰式按摩/SPA 以 800 THB/人估算。', 100),
        ('REF_ACTIVITY_COOKING', 'activity', 'cooking', 1500, '料理課以 1500 THB/人估算。', 120),
        ('REF_ACTIVITY_WATER', 'activity', 'water', 2500, '水上活動/潛水/跳島體驗以 2500 THB/人估算。', 130),
        ('REF_ACTIVITY_SHOW', 'activity', 'show', 1200, '表演/秀場活動以 1200 THB/人估算。', 140),
        ('REF_ACTIVITY_ADVENTURE', 'activity', 'adventure', 1800, '冒險活動以 1800 THB/人估算。', 150),
    ]
)

for view in [
    'member_a_attraction_records',
    'member_a_restaurant_records',
    'member_a_market_records',
    'member_a_activity_records',
    'member_a_record_export',
    'c_cost_coverage_summary',
]:
    conn.execute(f"DROP VIEW IF EXISTS {view}")

conn.execute("""
CREATE VIEW member_a_attraction_records AS
SELECT
    a.attraction_id AS data_id,
    'attraction' AS category,
    CASE
        WHEN a.attraction_name_zh IS NOT NULL AND a.attraction_name_en IS NOT NULL
             AND a.attraction_name_zh <> a.attraction_name_en
        THEN a.attraction_name_zh || ' / ' || a.attraction_name_en
        ELSE COALESCE(a.attraction_name_zh, a.attraction_name_en, '')
    END AS title,
    cm.city_name_en AS city,
    a.area_id AS area_id,
    COALESCE((
        SELECT COALESCE(c.typical_price, c.min_price, 0)
        FROM cost_item_master c
        WHERE c.related_attraction_id = a.attraction_id
        ORDER BY COALESCE(c.typical_price, c.min_price, 0) DESC
        LIMIT 1
    ), 0) AS estimated_cost_thb,
    CASE
        WHEN EXISTS (
            SELECT 1 FROM cost_item_master c
            WHERE c.related_attraction_id = a.attraction_id
              AND COALESCE(c.typical_price, c.min_price, 0) > 0
        )
        THEN 'direct_cost_item'
        ELSE 'free_or_missing_cost'
    END AS cost_quality,
    COALESCE((
        SELECT c.cost_estimation_note
        FROM cost_item_master c
        WHERE c.related_attraction_id = a.attraction_id
        ORDER BY COALESCE(c.typical_price, c.min_price, 0) DESC
        LIMIT 1
    ), '資料庫尚無獨立票價；可能免費、套票包含或需人工確認。') AS cost_note
FROM attraction_master a
LEFT JOIN city_master cm ON cm.city_id = a.city_id
""")

conn.execute("""
CREATE VIEW member_a_restaurant_records AS
SELECT
    r.restaurant_id AS data_id,
    'food' AS category,
    CASE
        WHEN r.restaurant_name_zh IS NOT NULL AND r.restaurant_name_en IS NOT NULL
             AND r.restaurant_name_zh <> r.restaurant_name_en
        THEN r.restaurant_name_zh || ' / ' || r.restaurant_name_en
        ELSE COALESCE(r.restaurant_name_zh, r.restaurant_name_en, '')
    END AS title,
    cm.city_name_en AS city,
    r.area_id AS area_id,
    CASE
        WHEN r.average_price_per_person LIKE '%-%'
        THEN (
            CAST(REPLACE(TRIM(SUBSTR(r.average_price_per_person, 1, INSTR(r.average_price_per_person, '-') - 1)), 'THB', '') AS REAL)
            +
            CAST(REPLACE(REPLACE(TRIM(SUBSTR(r.average_price_per_person, INSTR(r.average_price_per_person, '-') + 1)), 'THB', ''), ',', '') AS REAL)
        ) / 2.0
        ELSE CAST(REPLACE(REPLACE(COALESCE(r.average_price_per_person, '0'), 'THB', ''), ',', '') AS REAL)
    END AS estimated_cost_thb,
    'restaurant_average_price' AS cost_quality,
    '平均餐費估算，實際金額會依點餐內容變動。' AS cost_note
FROM restaurant_master r
LEFT JOIN city_master cm ON cm.city_id = r.city_id
""")

conn.execute("""
CREATE VIEW member_a_market_records AS
SELECT
    m.market_id AS data_id,
    'market' AS category,
    CASE
        WHEN m.market_name_zh IS NOT NULL AND m.market_name_en IS NOT NULL
             AND m.market_name_zh <> m.market_name_en
        THEN m.market_name_zh || ' / ' || m.market_name_en
        ELSE COALESCE(m.market_name_zh, m.market_name_en, '')
    END AS title,
    cm.city_name_en AS city,
    m.area_id AS area_id,
    CASE
        WHEN LOWER(COALESCE(m.market_type, '') || ' ' || COALESCE(m.opening_pattern, '') || ' ' || COALESCE(m.best_time_slot, '')) LIKE '%night%'
             OR COALESCE(m.best_time_slot, '') LIKE '%evening%'
        THEN 450
        WHEN LOWER(COALESCE(m.market_type, '') || ' ' || COALESCE(m.main_features, '')) LIKE '%shopping%'
             OR COALESCE(m.shopping_level, '') = 'high'
        THEN 500
        ELSE 300
    END AS estimated_cost_thb,
    'category_imputed' AS cost_quality,
    CASE
        WHEN LOWER(COALESCE(m.market_type, '') || ' ' || COALESCE(m.opening_pattern, '') || ' ' || COALESCE(m.best_time_slot, '')) LIKE '%night%'
             OR COALESCE(m.best_time_slot, '') LIKE '%evening%'
        THEN '夜市通常免門票，但晚餐、小吃與飲料抓 450 THB/人。'
        WHEN LOWER(COALESCE(m.market_type, '') || ' ' || COALESCE(m.main_features, '')) LIKE '%shopping%'
             OR COALESCE(m.shopping_level, '') = 'high'
        THEN '購物商場/市集抓餐飲與基本購物消費 500 THB/人；精品購物不含在內。'
        ELSE '市集/商圈通常免門票，但抓餐飲與小額購物基本消費 300 THB/人。'
    END AS cost_note
FROM market_master m
LEFT JOIN area_master ar ON ar.area_id = m.area_id
LEFT JOIN city_master cm ON cm.city_id = ar.city_id
""")

conn.execute("""
CREATE VIEW member_a_activity_records AS
SELECT
    a.activity_id AS data_id,
    'activity' AS category,
    CASE
        WHEN a.activity_name_zh IS NOT NULL AND a.activity_name_en IS NOT NULL
             AND a.activity_name_zh <> a.activity_name_en
        THEN a.activity_name_zh || ' / ' || a.activity_name_en
        ELSE COALESCE(a.activity_name_zh, a.activity_name_en, '')
    END AS title,
    cm.city_name_en AS city,
    a.area_id AS area_id,
    CASE
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%spa%'
          OR LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%massage%'
        THEN 800
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%cooking%'
        THEN 1500
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%water%'
          OR LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%diving%'
          OR a.activity_name_zh LIKE '%潛水%'
          OR a.activity_name_zh LIKE '%跳島%'
        THEN 2500
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%show%'
          OR a.activity_name_zh LIKE '%秀%'
          OR a.activity_name_zh LIKE '%表演%'
        THEN 1200
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%adventure%'
          OR a.activity_name_zh LIKE '%冒險%'
        THEN 1800
        ELSE 800
    END AS estimated_cost_thb,
    'category_imputed' AS cost_quality,
    CASE
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%spa%'
          OR LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%massage%'
        THEN '泰式按摩/SPA 以 800 THB/人估算。'
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%cooking%'
        THEN '料理課以 1500 THB/人估算。'
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%water%'
          OR LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%diving%'
          OR a.activity_name_zh LIKE '%潛水%'
          OR a.activity_name_zh LIKE '%跳島%'
        THEN '水上活動/潛水/跳島體驗以 2500 THB/人估算。'
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%show%'
          OR a.activity_name_zh LIKE '%秀%'
          OR a.activity_name_zh LIKE '%表演%'
        THEN '表演/秀場活動以 1200 THB/人估算。'
        WHEN LOWER(COALESCE(a.activity_type, '') || ' ' || COALESCE(a.short_description, '')) LIKE '%adventure%'
          OR a.activity_name_zh LIKE '%冒險%'
        THEN '冒險活動以 1800 THB/人估算。'
        ELSE '活動資料未建立明確票價時，以一般體驗活動 800 THB/人估算。'
    END AS cost_note
FROM activity_master a
LEFT JOIN area_master ar ON ar.area_id = a.area_id
LEFT JOIN city_master cm ON cm.city_id = ar.city_id
""")

conn.execute("""
CREATE VIEW member_a_record_export AS
SELECT * FROM member_a_attraction_records
UNION ALL
SELECT * FROM member_a_restaurant_records
UNION ALL
SELECT * FROM member_a_market_records
UNION ALL
SELECT * FROM member_a_activity_records
""")

conn.execute("""
CREATE VIEW c_cost_coverage_summary AS
SELECT
    category,
    COUNT(*) AS record_count,
    SUM(CASE WHEN estimated_cost_thb > 0 THEN 1 ELSE 0 END) AS priced_count,
    ROUND(100.0 * SUM(CASE WHEN estimated_cost_thb > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS priced_percent,
    SUM(CASE WHEN cost_quality LIKE '%imputed%' THEN 1 ELSE 0 END) AS imputed_count
FROM member_a_record_export
GROUP BY category
""")

conn.commit()

print('✅ views / data quality tables created')
display(pd.read_sql('SELECT * FROM c_cost_coverage_summary ORDER BY category', conn))


✅ View 建立完成：attraction_ticket_view / accommodation_cost_view


,item_name_zh,related_attraction_id,typical_price,unit
0,大皇宮外國成人門票,A001,500,per_person
1,鄭王廟外國成人門票,A004,200,per_person
2,臥佛寺外國成人門票,A003,300,per_person
3,真理寺日間成人門票,A041,500,per_person
4,Mahanakhon SkyWalk 白天門票估算,A014,1080,per_person


---
# Part 2：從 Excel 資料建立 RAG 向量庫

### 資料來源改變說明

每一筆景點 / 餐廳都會生成一段 chunk，並帶有 metadata（id、名稱、城市、區域）。

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


def df_to_documents(df, id_col, name_zh_col, name_en_col, city_col, area_col,
                    text_cols, doc_type='attraction'):
    """
    把 DataFrame 的每一筆轉成 LangChain Document。
    text_cols：要合併成一段描述文字的欄位清單。
    """
    docs = []
    for _, row in df.iterrows():
        # 把指定欄位的文字合併，中間用空格分隔，並過濾掉 NaN
        parts = []
        for col in text_cols:
            val = str(row.get(col, '')).strip()
            if val and val.lower() not in ('nan', 'none', 'not_applicable'):
                parts.append(val)
        content = ' '.join(parts)
        if not content:
            continue

        docs.append(Document(
            page_content=content,
            metadata={
                'id': str(row.get(id_col, '')),
                'name_zh': str(row.get(name_zh_col, '')),
                'name_en': str(row.get(name_en_col, '')),
                'city_id': str(row.get(city_col, '')),
                'area_id': str(row.get(area_col, '')),
                'type': doc_type,
            }
        ))
    return docs


# 景點文件
attraction_df = loaded_dfs.get('attraction_master', pd.DataFrame())
attraction_docs = df_to_documents(
    df=attraction_df,
    id_col='attraction_id',
    name_zh_col='attraction_name_zh',
    name_en_col='attraction_name_en',
    city_col='city_id',
    area_col='area_id',
    text_cols=[
        'attraction_name_zh', 'attraction_name_en',
        'short_description', 'main_features',
        'unique_value_for_agent', 'suitable_for',
        'travel_style_tags', 'recommended_reason',
    ],
    doc_type='attraction'
)

# 餐廳文件
restaurant_df = loaded_dfs.get('restaurant_master', pd.DataFrame())
restaurant_docs = df_to_documents(
    df=restaurant_df,
    id_col='restaurant_id',
    name_zh_col='restaurant_name_zh',
    name_en_col='restaurant_name_en',
    city_col='city_id',
    area_col='area_id',
    text_cols=[
        'restaurant_name_zh', 'restaurant_name_en',
        'short_description', 'signature_dishes',
        'main_features', 'recommended_reason',
        'suitable_for', 'dining_situation_tags',
    ],
    doc_type='restaurant'
)

all_raw_docs = attraction_docs + restaurant_docs
print(f'✅ 文件生成完成：景點 {len(attraction_docs)} 筆 + 餐廳 {len(restaurant_docs)} 筆 = 共 {len(all_raw_docs)} 筆')

/tmp/ipykernel_6245/3967088936.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ 文件生成完成：景點 80 筆 + 餐廳 80 筆 = 共 160 筆


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
)

split_docs = text_splitter.split_documents(all_raw_docs)
print(f'✅ 切塊完成：{len(all_raw_docs)} 筆 → {len(split_docs)} 個片段')
print('\n── 範例片段 ──')
if split_docs:
    print('內容：', split_docs[0].page_content[:100])
    print('Metadata：', split_docs[0].metadata)

✅ 切塊完成：160 筆 → 160 個片段

── 範例片段 ──
內容： 大皇宮 The Grand Palace 曼谷皇室建築群核心，集合宮殿、寺廟與傳統泰式建築細節。 皇室建築群；玉佛寺；金色尖塔與壁畫；老城區文化路線 判斷曼谷文化行程核心點，需結合服裝規範與高溫人潮限
Metadata： {'id': 'A001', 'name_zh': '大皇宮', 'name_en': 'The Grand Palace', 'city_id': 'C001', 'area_id': 'AREA004', 'type': 'attraction'}


In [10]:
print('⏳ 載入 multilingual-e5-small（首次約需 1~2 分鐘下載）...')

embedding_model = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-small',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print('⏳ 建立 FAISS 索引...')
vectorstore = FAISS.from_documents(split_docs, embedding_model)

FAISS_INDEX_PATH = '/content/thailand_faiss_index'
vectorstore.save_local(FAISS_INDEX_PATH)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print(f'✅ FAISS 索引建立完成，已儲存至：{FAISS_INDEX_PATH}')

⏳ 載入 multilingual-e5-small（首次約需 1~2 分鐘下載）...


/tmp/ipykernel_6245/2832145448.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

⏳ 建立 FAISS 索引...
✅ FAISS 索引建立完成，已儲存至：/content/thailand_faiss_index


In [11]:
# ── 測試 RAG 檢索 ─────────────────────────────────────────────
test_queries = [
    '曼谷有什麼文化景點？',
    '清邁適合親子的餐廳',
    '普吉島日落觀景點',
    '米其林餐廳推薦',
]

for q in test_queries:
    results = retriever.invoke(q)
    print(f'\n🔍 {q}')
    for i, doc in enumerate(results, 1):
        name = doc.metadata.get('name_zh', '?')
        doc_type = doc.metadata.get('type', '?')
        print(f'   [{i}] [{doc_type}] {name} | {doc.page_content[:50]}...')


🔍 曼谷有什麼文化景點？
   [1] [attraction] 大皇宮 | 大皇宮 The Grand Palace 曼谷皇室建築群核心，集合宮殿、寺廟與傳統泰式建築細節。 皇...
   [2] [attraction] 大城歷史公園 | 大城歷史公園 Ayutthaya Historical Park 保存阿瑜陀耶王朝寺廟與宮殿遺址的世...
   [3] [attraction] 曼谷國家博物館 | 曼谷國家博物館 Bangkok National Museum 收藏泰國歷史、宗教藝術與皇室文物的大...

🔍 清邁適合親子的餐廳
   [1] [restaurant] Ginger Farm Kitchen | Ginger Farm Kitchen Ginger Farm Kitchen Ginger Far...
   [2] [restaurant] Railay Family Restaurant | Railay Family Restaurant Railay Family Restaurant ...
   [3] [restaurant] Barrab Restaurant | Barrab Restaurant Barrab Restaurant Barrab Restaur...

🔍 普吉島日落觀景點
   [1] [attraction] 神仙半島 | 神仙半島 Promthep Cape 普吉南端海岬觀景點，以日落與海岸線視野聞名。 日落；海岬步道；...
   [2] [attraction] 詹姆士龐德島 | 詹姆士龐德島 James Bond Island / Ko Tapu 攀牙灣石灰岩海上地形代表景點，...
   [3] [attraction] 普吉老鎮 | 普吉老鎮 Phuket Old Town 中葡式建築、壁畫、咖啡店與週日市集構成城市漫遊區。 中葡建...

🔍 米其林餐廳推薦
   [1] [restaurant] Dash Restaurant and Bar | Dash Restaurant and Bar Dash Restaurant and Bar Da...
   [2] [restaurant] PRU | PRU PRU PRU位於Phuket的Cherngtalay / Trisara

---
# Part 3：SQLite FunctionTool 串接 ADK Agent

In [ ]:

import sqlite3
import json
import pandas as pd
import re

def get_db():
    """取得 SQLite 連線。"""
    return sqlite3.connect(str(DB_PATH))


def query_attraction_cost(attraction_name: str, adult_count: int = 1, child_count: int = 0) -> str:
    """查詢景點票價。

    修正版：目前 DB 沒有 cost_item_attraction_map，因此直接用
    cost_item_master.related_attraction_id 對應 attraction_master.attraction_id。
    """
    conn = get_db()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT attraction_id, attraction_name_zh, attraction_name_en,
               city_id, area_id, recommended_duration, budget_level
        FROM attraction_master
        WHERE attraction_name_zh LIKE ? OR attraction_name_en LIKE ?
        LIMIT 1
    """, (f"%{attraction_name}%", f"%{attraction_name}%"))
    attr_row = cursor.fetchone()

    if not attr_row:
        conn.close()
        return json.dumps({"error": f"找不到景點「{attraction_name}」，請確認名稱。"}, ensure_ascii=False)

    attr_id, name_zh, name_en, city_id, area_id, duration, budget_level = attr_row

    cursor.execute("""
        SELECT item_name_zh, min_price, typical_price, max_price,
               COALESCE(unit, 'per_person') AS unit,
               cost_estimation_note
        FROM cost_item_master
        WHERE related_attraction_id = ?
        ORDER BY COALESCE(typical_price, min_price, 0) DESC
        LIMIT 3
    """, (attr_id,))
    cost_rows = cursor.fetchall()
    conn.close()

    if cost_rows:
        ticket_adult = float(cost_rows[0][2] or cost_rows[0][1] or 0)
        ticket_child = 0.0
        for row in cost_rows:
            label = str(row[0]).lower()
            if "兒童" in label or "child" in label:
                ticket_child = float(row[2] or row[1] or 0)
                break
        total = ticket_adult * adult_count + ticket_child * child_count
        ticket_detail = [
            {"票種": r[0], "最低": r[1], "典型": r[2], "最高": r[3], "單位": r[4], "備註": r[5]}
            for r in cost_rows
        ]
        cost_quality = "direct_cost_item"
    else:
        total = 0.0
        ticket_detail = [{"備註": "資料庫尚未收錄此景點票價；可能免費、套票包含或需現場確認。"}]
        cost_quality = "free_or_missing_cost"

    return json.dumps({
        "景點名稱": name_zh,
        "英文名稱": name_en,
        "城市": city_id,
        "建議遊覽時間": duration,
        "預算等級": budget_level,
        "票價明細": ticket_detail,
        "計算人數": {"成人": adult_count, "兒童": child_count},
        "預估總票價_THB": total,
        "cost_quality": cost_quality,
    }, ensure_ascii=False, indent=2)


def query_member_a_record_export(city_name: str = "", category: str = "", limit: int = 20) -> str:
    """給 A / B 測試用：查詢已補 estimated_cost_thb 的候選資料。"""
    conn = get_db()
    sql = """
        SELECT data_id, category, title, city, area_id,
               estimated_cost_thb, cost_quality, cost_note
        FROM member_a_record_export
        WHERE 1=1
    """
    params = []
    if city_name:
        sql += " AND LOWER(city) = LOWER(?)"
        params.append(city_name)
    if category:
        sql += " AND category = ?"
        params.append(category)
    sql += " ORDER BY category, estimated_cost_thb DESC LIMIT ?"
    params.append(limit)

    df = pd.read_sql(sql, conn, params=params)
    conn.close()
    return df.to_json(orient="records", force_ascii=False, indent=2)


def query_cost_coverage_summary() -> str:
    """檢查各類資料目前有多少筆可提供費用。"""
    conn = get_db()
    df = pd.read_sql("SELECT * FROM c_cost_coverage_summary ORDER BY category", conn)
    conn.close()
    return df.to_json(orient="records", force_ascii=False, indent=2)


def query_accommodation_cost(city_id: str, budget_level: str = '') -> str:
    """查詢特定城市住宿費用選項。"""
    conn = get_db()

    if budget_level:
        level_map = {'低': 'low', '中': 'medium', '高': 'high', '奢華': 'luxury',
                     'low': 'low', 'medium': 'medium', 'high': 'high', 'luxury': 'luxury'}
        en_level = level_map.get(budget_level, budget_level)
        df = pd.read_sql("""
            SELECT item_name_zh, item_name_en, min_price, typical_price, max_price,
                   unit, budget_level, suitable_user_types, cost_estimation_note
            FROM accommodation_cost_view
            WHERE (city_id = ? OR city_id = 'C020') AND budget_level LIKE ?
            ORDER BY typical_price ASC
        """, conn, params=(city_id, f'%{en_level}%'))
    else:
        df = pd.read_sql("""
            SELECT item_name_zh, item_name_en, min_price, typical_price, max_price,
                   unit, budget_level, suitable_user_types, cost_estimation_note
            FROM accommodation_cost_view
            WHERE city_id = ? OR city_id = 'C020'
            ORDER BY typical_price ASC
        """, conn, params=(city_id,))
    conn.close()

    if df.empty:
        return json.dumps({'error': f'找不到城市 {city_id} 的住宿費用資料'}, ensure_ascii=False)

    return json.dumps({
        '城市代碼': city_id,
        '住宿選項': df.head(10).to_dict(orient='records')
    }, ensure_ascii=False, indent=2)


def query_restaurant_options(city_id: str, cuisine_type: str = '', budget_level: str = '') -> str:
    """查詢餐廳選項。"""
    conn = get_db()
    sql = """
        SELECT restaurant_name_zh, restaurant_name_en, food_category, cuisine_type,
               dining_style, average_price_per_person, budget_level,
               best_meal_time, recommended_reason, source_url
        FROM restaurant_master
        WHERE city_id = ?
    """
    params = [city_id]

    if cuisine_type:
        sql += " AND (cuisine_type LIKE ? OR food_category LIKE ?)"
        params.extend([f'%{cuisine_type}%', f'%{cuisine_type}%'])
    if budget_level:
        sql += " AND budget_level LIKE ?"
        params.append(f'%{budget_level}%')

    sql += " ORDER BY restaurant_id LIMIT 10"
    df = pd.read_sql(sql, conn, params=params)
    conn.close()

    if df.empty:
        return json.dumps({'error': f'找不到城市 {city_id} 的餐廳資料'}, ensure_ascii=False)

    return df.to_json(orient='records', force_ascii=False, indent=2)


def calculate_trip_budget(
    days: int,
    adults: int,
    children: int = 0,
    city_id: str = 'C001',
    hotel_budget_level: str = '中',
    meal_budget_per_person_per_day: float = 500,
    attraction_list: str = ''
) -> str:
    """計算粗略旅行預算。"""
    people = adults + children
    nights = max(days - 1, 0)

    # hotel: use first matching option
    hotel_data = json.loads(query_accommodation_cost(city_id, hotel_budget_level))
    hotel_price = 0
    if isinstance(hotel_data, dict) and hotel_data.get('住宿選項'):
        hotel_price = float(hotel_data['住宿選項'][0].get('typical_price') or 0)

    hotel_total = hotel_price * nights
    meal_total = meal_budget_per_person_per_day * people * days

    attraction_total = 0
    attraction_details = []
    for name in [x.strip() for x in attraction_list.split(',') if x.strip()]:
        data = json.loads(query_attraction_cost(name, adult_count=adults, child_count=children))
        price = float(data.get('預估總票價_THB') or 0)
        attraction_total += price
        attraction_details.append({
            'name': name,
            'estimated_total_thb': price,
            'cost_quality': data.get('cost_quality', '')
        })

    total = hotel_total + meal_total + attraction_total

    return json.dumps({
        'days': days,
        'nights': nights,
        'people': people,
        'hotel_total_THB': hotel_total,
        'meal_total_THB': meal_total,
        'attraction_total_THB': attraction_total,
        'attraction_details': attraction_details,
        'estimated_trip_total_THB': total,
        'note': '交通、購物、保險與機票未包含；市集/活動可透過 member_a_record_export 的 estimated_cost_thb 另行加總。'
    }, ensure_ascii=False, indent=2)


✅ FunctionTool 函式定義完成（4 個工具）


In [14]:

print('=== 測試 1：查詢大皇宮票價（2大1小）===')
print(query_attraction_cost('大皇宮', adult_count=2, child_count=1))

print('\n=== 測試 2：曼谷中等住宿選項 ===')
print(query_accommodation_cost('C001', budget_level='中'))

print('\n=== 測試 3：清邁泰式餐廳 ===')
print(query_restaurant_options('C002', cuisine_type='泰式'))

print('\n=== 測試 4：資料成本覆蓋率 ===')
print(query_cost_coverage_summary())

print('\n=== 測試 5：Bangkok activity / market export ===')
print(query_member_a_record_export(city_name='Bangkok', category='activity', limit=5))


=== 測試 1：查詢大皇宮票價（2大1小）===
{
  "景點名稱": "大皇宮",
  "英文名稱": "The Grand Palace",
  "城市": "C001",
  "建議遊覽時間": "2.5-3.5 小時",
  "預算等級": "高",
  "票價明細": [
    {
      "票種": "大皇宮外國成人門票",
      "最低": 500,
      "典型": 500,
      "最高": 500,
      "單位": "per_person"
    }
  ],
  "計算人數": {
    "成人": 2,
    "兒童": 1
  },
  "預估總票價_THB": 1000
}

=== 測試 2：曼谷中等住宿選項 ===
{"error": "找不到城市 C001 的住宿費用資料"}

=== 測試 3：清邁泰式餐廳 ===
{
  "城市代碼": "C002",
  "餐廳列表": [
    {
      "restaurant_name_zh": "Ginger Farm Kitchen",
      "restaurant_name_en": "Ginger Farm Kitchen",
      "food_category": "親子友善/有機餐廳",
      "cuisine_type": "泰式與農場料理",
      "average_price_per_person": "300-900 THB",
      "budget_level": "中",
      "recommended_reason": "親子與健康取向旅客的清邁餐廳選項；招牌如 有機泰式料理 能讓行程不是只停留在景點清單，而是加入明確餐飲記憶點。",
      "suitable_for": "家庭; 親子; 健康飲食者",
      "best_meal_time": "午餐：親子行程較穩定",
      "area_id": "AREA012"
    },
    {
      "restaurant_name_zh": "The House by Ginger",
      "restaurant_name_en": "The House by Ginger",
      "

In [15]:

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import FunctionTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

BUDGET_TOOLS = [
    FunctionTool(query_attraction_cost),
    FunctionTool(query_accommodation_cost),
    FunctionTool(query_restaurant_options),
    FunctionTool(calculate_trip_budget),
    FunctionTool(query_member_a_record_export),
    FunctionTool(query_cost_coverage_summary),
]

CITY_CODE_MAP = """
城市代碼對照：
- C001 = 曼谷 (Bangkok)
- C002 = 清邁 (Chiang Mai)
- C003 = 清萊 (Chiang Rai)
- C004 = 普吉 (Phuket)
- C005 = 芭達雅 (Pattaya)
- C006 = 華欣 (Hua Hin)
- C007 = 大城 (Ayutthaya)
- C008 = 甲米 (Krabi)
- C009 = 蘇美島 (Koh Samui)
"""

budget_agent = LlmAgent(
    name='BudgetCalculatorAgent',
    model=LiteLlm(model=GROQ_MODEL),
    instruction=f"""
你是泰國旅行行程小幫手中的「精算師 Agent」。
你的唯一職責是協助使用者計算旅遊費用。

{CITY_CODE_MAP}

規則：
1. 遇到費用、票價、預算相關問題，必須先呼叫工具查詢資料庫，不可自行猜測。
2. 景點票價優先使用 query_attraction_cost。
3. 住宿費用使用 query_accommodation_cost。
4. 餐廳使用 query_restaurant_options。
5. 若要給 A 端候選資料，使用 query_member_a_record_export，因為它包含 estimated_cost_thb。
6. 所有費用單位為泰銖（THB），回答要標明。
7. 清楚列出費用分項讓使用者理解。
8. 若 cost_quality 是 category_imputed，要說明是估算值，不是官方票價。
9. 使用繁體中文回答。
""",
    tools=BUDGET_TOOLS,
)

print(f'✅ 精算 Agent 建立完成，掛載 {len(BUDGET_TOOLS)} 個工具')


✅ 精算 Agent 建立完成，掛載 4 個工具


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [16]:
# ── 端對端測試 ────────────────────────────────────────────────
import asyncio

async def run_budget_agent(query: str, session_id: str = 'test') -> str:
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name='thailand_trip', user_id='user_test', session_id=session_id
    )
    runner = Runner(
        agent=budget_agent, app_name='thailand_trip',
        session_service=session_service
    )
    content = Content(role='user', parts=[Part(text=query)])
    final_text = ''
    async for event in runner.run_async(
        user_id='user_test', session_id=session.id, new_message=content
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_text = '\n'.join(
                    p.text for p in event.content.parts if getattr(p, 'text', None)
                )
            break
    return final_text

test_q = '我們 2 個大人帶 1 個小孩去曼谷 4 天，想去大皇宮和恰圖恰市集，住中等飯店，每天餐飲每人 500 泰銖，請估算總費用。'
print(f'🧳 使用者：{test_q}\n')
response = await run_budget_agent(test_q)
print(f'💰 精算 Agent 回答：\n{response}')

🧳 使用者：我們 2 個大人帶 1 個小孩去曼谷 4 天，想去大皇宮和恰圖恰市集，住中等飯店，每天餐飲每人 500 泰銖，請估算總費用。



03:37:27 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
03:37:27 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


💰 精算 Agent 回答：
根據您的需求，前往曼谷 4 天的預估總費用為 15000.0 泰銖，包括住宿費用 8000 泰銖、餐飲費用 6000 泰銖、以及景點門票費用 1000 泰銖。其中，恰圖恰市集的門票價格因資料庫查無資料，故無法估算，建議您到現場確認。另外，這個估算不包含國際機票、島內交通、購物以及小費等其他費用。


---
# Part 4：壓力與品質測試

In [17]:
import time
import asyncio
import re

MODEL_NAME_MAP = {
    GROQ_MODEL: 'BudgetAgent_70b',
    GROQ_MODEL_LIGHT: 'BudgetAgent_8b',
}

async def run_agent_with_retry(query: str, session_id: str, max_retries: int = 3, force_model: str = None) -> str:
    """
    force_model：若指定，只用該模型，不做 fallback。
    """
    models_to_try = [force_model] if force_model else [GROQ_MODEL, GROQ_MODEL_LIGHT]

    for model_str in models_to_try:
        agent = LlmAgent(
            name=MODEL_NAME_MAP[model_str],
            model=LiteLlm(model=model_str),
            instruction=budget_agent.instruction,
            tools=BUDGET_TOOLS,
        )

        for attempt in range(1, max_retries + 1):
            try:
                session_service = InMemorySessionService()
                session = await session_service.create_session(
                    app_name='thailand_trip',
                    user_id='user_test',
                    session_id=f"{session_id}_{model_str[-3:]}_{attempt}"
                )
                runner = Runner(
                    agent=agent,
                    app_name='thailand_trip',
                    session_service=session_service
                )
                content = Content(role='user', parts=[Part(text=query)])
                final_text = ''
                async for event in runner.run_async(
                    user_id='user_test',
                    session_id=session.id,
                    new_message=content
                ):
                    if event.is_final_response():
                        if event.content and event.content.parts:
                            final_text = '\n'.join(
                                p.text for p in event.content.parts
                                if getattr(p, 'text', None)
                            )
                        break
                return final_text

            except Exception as e:
                err_str = str(e)
                if 'rate_limit_exceeded' in err_str or 'RateLimitError' in err_str:
                    match = re.search(r'try again in ([\d.]+)s', err_str)
                    wait_sec = float(match.group(1)) + 3 if match else 20
                    print(f'   ⚠️ [{MODEL_NAME_MAP[model_str]}] Attempt {attempt} Rate Limit，等待 {wait_sec:.0f} 秒後 retry...')
                    time.sleep(wait_sec)
                else:
                    raise e

        print(f'   ⚠️ {MODEL_NAME_MAP[model_str]} 已用完 {max_retries} 次 retry，切換下一個模型...')

    return '❌ 所有模型都超過 Rate Limit，請稍後再試'



TEST_CASES = [
    {'id': 'TC-01', 'desc': '正常：2人3天曼谷行',
     'query': '2個大人去曼谷3天，想參觀大皇宮跟恰圖恰市集，住中等飯店，估算總費用。',
     'force_model': None},           # 預設：70b 優先，超限才降 8b
    {'id': 'TC-02', 'desc': '多人：4人5天清邁行',
     'query': '4個大人去清邁5天，想去雙龍寺，每天餐飲每人400泰銖，住中等飯店，估算費用。',
     'force_model': None},
    {'id': 'TC-03', 'desc': '邊界：查不存在景點',
     'query': '我想去曼谷的夢幻世界樂園，2個大人票價多少？',
     'force_model': GROQ_MODEL},     # 固定用 70b，避免 8b tool call 格式錯誤
    {'id': 'TC-04', 'desc': '餐廳查詢：普吉海鮮',
     'query': '普吉島有什麼海鮮餐廳推薦？預算中等。',
     'force_model': None},
]

DELAY_BETWEEN_CASES = 20
results = []

for tc in TEST_CASES:
    print(f"\n{'='*55}")
    print(f"🧪 {tc['id']}：{tc['desc']}")
    start = time.time()
    try:
        resp = await run_agent_with_retry(
            tc['query'],
            session_id=f"stress_{tc['id']}",
            force_model=tc.get('force_model')
        )
        elapsed = time.time() - start
        status = '✅ PASS'
        print(f'回答：{resp[:150]}...')
    except Exception as e:
        elapsed = time.time() - start
        status = f'❌ FAIL: {e}'
        print(status)

    results.append({
        'ID': tc['id'],
        '描述': tc['desc'],
        '狀態': status,
        '耗時(秒)': round(elapsed, 1)
    })

    if tc != TEST_CASES[-1]:
        print(f'⏳ 等待 {DELAY_BETWEEN_CASES} 秒...')
        time.sleep(DELAY_BETWEEN_CASES)

print('\n📊 壓力測試結果')
display(pd.DataFrame(results))


🧪 TC-01：正常：2人3天曼谷行
回答：根據估算，2個大人去曼谷3天，參觀大皇宮和恰圖恰市集，住中等飯店的總費用約為10,000泰銖，包含住宿費6,000泰銖、餐飲費3,000泰銖和景點費用1,000泰銖。注意這個費用不包含國際機票、島內交通、購物和小費等費用。另外，由於恰圖恰市集的費用查無資料，建議到現場確認。...
⏳ 等待 20 秒...

🧪 TC-02：多人：4人5天清邁行
回答：根據估算，4個大人去清邁5天，總費用約為18,000泰銖。其中包含：

*   住宿費用：10,000泰銖
*   餐飲費用：8,000泰銖
*   景點費用：由於資料庫尚未收錄雙龍寺票價，建議您到現場確認。

備註：此估算不含國際機票、島內交通、購物和小費。...
⏳ 等待 20 秒...

🧪 TC-03：邊界：查不存在景點

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

   ⚠️ [BudgetAgent_70b] Attempt 1 Rate Limit，等待 20 秒後 retry...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

   ⚠️ [BudgetAgent_70b] Attempt 2 Rate Limit，等待 37 秒後 retry...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

   ⚠️ [BudgetAgent_70b] Attempt 3 Rate Limit，等待 4

,ID,描述,狀態,耗時(秒)
0,TC-01,正常：2人3天曼谷行,✅ PASS,1.0
1,TC-02,多人：4人5天清邁行,✅ PASS,1.0
2,TC-03,邊界：查不存在景點,✅ PASS,97.4
3,TC-04,餐廳查詢：普吉海鮮,✅ PASS,61.6


In [18]:
# ── RAG 品質測試（基於真實資料）──────────────────────────────
print('=== RAG 檢索品質測試（真實資料）===')

rag_tests = [
    '曼谷老城區有什麼文化景點',
    '清邁咖啡廳推薦',
    '普吉島適合親子的海灘',
    '泰式炒麵在哪裡吃',
    '芭達雅夜生活景點',
]

for q in rag_tests:
    docs = retriever.invoke(q)
    print(f'\n🔍 {q}')
    for i, doc in enumerate(docs, 1):
        name = doc.metadata.get('name_zh', '?')
        dtype = doc.metadata.get('type', '?')
        city = doc.metadata.get('city_id', '?')
        print(f'   [{i}] [{dtype}] {city} | {name}')

=== RAG 檢索品質測試（真實資料）===

🔍 曼谷老城區有什麼文化景點
   [1] [attraction] C001 | 大皇宮
   [2] [attraction] C001 | 金山寺
   [3] [attraction] C007 | 大城歷史公園

🔍 清邁咖啡廳推薦
   [1] [restaurant] C003 | Chivit Thamma Da 河畔咖啡
   [2] [restaurant] C002 | Mae Sai 咖哩麵
   [3] [restaurant] C002 | Huen Phen 北部泰菜

🔍 普吉島適合親子的海灘
   [1] [attraction] C004 | 卡塔海灘
   [2] [attraction] C004 | 芭東海灘
   [3] [attraction] C005 | 中天海灘

🔍 泰式炒麵在哪裡吃
   [1] [restaurant] C005 | 芭達雅水上市場美食區
   [2] [restaurant] C001 | Thipsamai Pad Thai
   [3] [restaurant] C001 | 朱拉豬肉媽媽麵

🔍 芭達雅夜生活景點
   [1] [attraction] C005 | 步行街
   [2] [attraction] C005 | 東芭樂園
   [3] [attraction] C005 | 真理寺


---
# 給成員 A 的串接說明

成員 A 可以直接使用以下兩個物件：

```python
# 1. SQLite FunctionTool 清單（精算 Agent 使用）
BUDGET_TOOLS  # List[FunctionTool]，包含 4 個查詢函式

# 2. RAG Retriever（導遊 Agent 使用）
retriever     # FAISS retriever，輸入問題字串，輸出 3 個最相關景點/餐廳文件

# 包裝成 FunctionTool 的範例：
def rag_search_attractions(query: str) -> str:
    '''搜尋景點或餐廳相關資訊，適合回答「有什麼好玩的」這類問題。'''
    docs = retriever.invoke(query)
    return '\n---\n'.join(
        f"{d.metadata.get('name_zh')}：{d.page_content[:200]}" for d in docs
    )
```

### 城市代碼速查
| 代碼 | 城市 | 代碼 | 城市 |
|------|------|------|------|
| C001 | 曼谷 | C005 | 芭達雅 |
| C002 | 清邁 | C006 | 華欣 |
| C003 | 清萊 | C007 | 大城 |
| C004 | 普吉 | C008 | 甲米 |